# Лаба 2

**Тема:** обработка пропусков, кодирование категориальных признаков, масштабирование данных.

**Датасет:** Adult Income (`adult.csv`).

## Цель
Подготовить данные для дальнейшего построения моделей машинного обучения:
- обработать пропуски;
- закодировать категориальные признаки;
- масштабировать числовые признаки.

In [2]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
# Загрузка данных

df = pd.read_csv("adult.csv")
print("Размер датасета:", df.shape)
df.head()

Размер датасета: (32561, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [4]:
# Первичный анализ: типы данных и пропуски

# В этом наборе пропуски закодированы как '?', заменим их на NaN
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].str.strip()

df = df.replace("?", np.nan)

print("\nТипы столбцов:")
print(df.dtypes)

print("\nКоличество пропусков по столбцам:")
missing = df.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])


Типы столбцов:
age               int64
workclass           str
fnlwgt            int64
education           str
education.num     int64
marital.status      str
occupation          str
relationship        str
race                str
sex                 str
capital.gain      int64
capital.loss      int64
hours.per.week    int64
native.country      str
income              str
dtype: object

Количество пропусков по столбцам:
occupation        1843
workclass         1836
native.country     583
dtype: int64


In [6]:
# Разделим признаки на X и целевую переменную y
X = df.drop(columns=["income"])
y = df["income"]

# Определим списки числовых и категориальных признаков
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()

print("Числовые признаки:", numeric_features)
print("Категориальные признаки:", categorical_features)

# Пайплайн для числовых признаков: заполнение медианой + стандартизация
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Пайплайн для категориальных признаков: заполнение модой + one-hot кодирование
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# Объединяем преобразования
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Применяем preprocessing
X_processed = preprocessor.fit_transform(X)
print("Форма матрицы после preprocessing:", X_processed.shape)

Числовые признаки: ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
Категориальные признаки: ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country']
Форма матрицы после preprocessing: (32561, 105)


In [7]:
# Получим названия новых признаков после one-hot кодирования
feature_names = preprocessor.get_feature_names_out()

# Преобразуем результат в DataFrame для удобства просмотра
X_processed_df = pd.DataFrame(
    X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed,
    columns=feature_names,
)

# Добавим целевую переменную
final_df = X_processed_df.copy()
final_df["income"] = y.values

print("Итоговый размер подготовленного датасета:", final_df.shape)
final_df.head()

Итоговый размер подготовленного датасета: (32561, 106)


,num__age,num__fnlwgt,num__education.num,num__capital.gain,num__capital.loss,num__hours.per.week,cat__workclass_Federal-gov,cat__workclass_Local-gov,cat__workclass_Never-worked,cat__workclass_Private,...,cat__native.country_Puerto-Rico,cat__native.country_Scotland,cat__native.country_South,cat__native.country_Taiwan,cat__native.country_Thailand,cat__native.country_Trinadad&Tobago,cat__native.country_United-States,cat__native.country_Vietnam,cat__native.country_Yugoslavia,income
0,3.769612,-1.067997,-0.420060,-0.14592,10.593507,-0.035429,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,<=50K
1,3.183112,-0.539169,-0.420060,-0.14592,10.593507,-1.817204,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,<=50K
2,2.010110,-0.035220,-0.031360,-0.14592,10.593507,-0.035429,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,<=50K
3,1.130359,-0.468215,-2.363558,-0.14592,9.461864,-0.035429,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,<=50K
4,0.177296,0.709482,-0.031360,-0.14592,9.461864,-0.035429,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,<=50K


In [8]:
# Сохраним итоговый набор данных
final_df.to_csv("adult_preprocessed.csv", index=False)
print("Файл сохранен: adult_preprocessed.csv")

print("\nКраткий вывод:")
print("1) Пропуски ('?') заменены и обработаны через SimpleImputer.")
print("2) Категориальные признаки закодированы OneHotEncoder.")
print("3) Числовые признаки масштабированы StandardScaler.")

Файл сохранен: adult_preprocessed.csv

Краткий вывод:
1) Пропуски ('?') заменены и обработаны через SimpleImputer.
2) Категориальные признаки закодированы OneHotEncoder.
3) Числовые признаки масштабированы StandardScaler.


$$
z = \frac{x - \mu}{\sigma}
$$

- $x$ - исходное значение признака;
- $\mu$ - среднее значение признака по выборке;
- $\sigma$ - стандартное отклонение признака;
- $z$ - масштабированное значение.

После такого преобразования признаки имеют среднее, близкое к 0, и стандартное отклонение, близкое к 1. Это делает признаки сопоставимыми по масштабу и улучшает работу алгоритмов, чувствительных к диапазонам значений (например, логистическая регрессия, SVM, kNN).


После этого:
    среднее примерно 0,
    разброс примерно 1.
    
Пример (условно для age):
    было: 18, 45, 70
    стало: -1.1, 0.2, 1.4

Зачем:
чтобы признаки с большими числами (fnlwgt) не “давили” признаки с маленькими (age);
многие модели работают стабильнее и точнее на масштабированных данных.